In [1]:
from qick import *
# %matplotlib widget
%matplotlib notebook
# %matplotlib inline
import matplotlib.pyplot as plt

In [2]:
import numpy as np
from numpy.polynomial import Polynomial
import matplotlib.ticker as mtick
from matplotlib.ticker import MultipleLocator
# from tqdm import tqdm
from tqdm.notebook import tqdm
import xarray as xr

In [3]:
import os
import sys
sys.path.insert(0, '../../pattern/')
sys.path.insert(0, '../../instrument/')

In [4]:
from pathlib import Path

folder_name = Path.cwd().name
data_dir = Path("Z:/labdata/qcdlabs") / folder_name
data_dir.mkdir(parents=True, exist_ok=True)

In [5]:
sys.path.insert(0, '../../pattern/')
from helper_sweep import do_sweep

In [6]:
from xilinx_qick.class_drx import drx
from xilinx_qick.class_rox import rox
from xilinx_qick.class_sweep import sweep
from xilinx_qick.instr_xilinx_v1 import XilinxProg

xilinx_1 = XilinxProg(ip_address="10.0.100.21", mode='AveragerProgram')

Pyro.NameServer PYRO:Pyro.NameServer@0.0.0.0:8888
rfsoc4x2_1 PYRO:obj_2fce009e709b4ba49061697c43bba7e4@10.0.100.21:40291
QICK running on RFSoC4x2, software version 0.2.381

Firmware configuration (built Wed Sep  6 18:49:29 2023):

	Global clocks (MHz): tProc dispatcher timing 409.600, RF reference 491.520
	Groups of related clocks: [tProc clock, DAC tile 0], [DAC tile 2], [ADC tile 0]

	2 signal generator channels:
	0:	axis_signal_gen_v6 - fs=9830.400 Msps, fabric=614.400 MHz
		envelope memory: 65536 complex samples (6.667 us)
		32-bit DDS, range=9830.400 MHz
		DAC tile 0, blk 0 is DAC_B
	1:	axis_signal_gen_v6 - fs=9830.400 Msps, fabric=614.400 MHz
		envelope memory: 65536 complex samples (6.667 us)
		32-bit DDS, range=9830.400 MHz
		DAC tile 2, blk 0 is DAC_A

	2 readout channels:
	0:	axis_readout_v2 - configured by PYNQ
		fs=4423.680 Msps, decimated=552.960 MHz, 32-bit DDS, range=4423.680 MHz
		axis_avg_buffer v1.0 (no edge counter, no weights)
		memory 16384 accumulated, 1024 decima

In [7]:
xilinx_1.reps = int(1)
dr_ch0 = 0
ro_ch0 = 0

In [8]:
def set_if_frequency(xilinx_1, if_frequency_hz=0):
    dt = 10/9830.4
    # t_gen = np.arange(0, 1.6*2, dt)
    # s_gen = 1 * np.ones(len(t_gen))
    # s_gen[0] = 0
    # s_gen[-1] = 0

    t_gen = np.array([0, 0.1, 0.8, 1.5, 1.6])*2
    s_gen = np.array([0, 1, 1, 1, 0])

    dr_readout1 = drx(soc=xilinx_1.soccfg,
                    dr_ch=dr_ch0, ro_ch=ro_ch0, frequency= if_frequency_hz / 1e6, gain=0.9, phase=0)

    dr_readout1.wave.add(name='x2', t_data=t_gen, s_data=s_gen, idx=-1, interp_order=1)
    dr_readout1.rox.set(length=1.6, delay=0.4, sleep=1)
    xilinx_1.add(dr_readout1=dr_readout1)

xilinx_1.register_sweep('if_frequency_hz', set_if_frequency)

In [ ]:
plt.figure()
plt.plot(xilinx_1.config['dr_readout1'].wave.items[0].i_data, '.-')
plt.plot(xilinx_1.config['dr_readout1'].wave.items[0].q_data, '.-')
plt.show()

In [9]:
def set_reps(xilinx_1, reps=0):
    pass

xilinx_1.register_sweep('reps', set_reps)

In [ ]:
0.0125e6*20+8.08e9

In [ ]:
8.08e9 + 2e6

In [62]:
if_frequency_list = np.arange(8.08e9, 8.080000005e9+1, 0.0002e3)
print(len(if_frequency_list))

reps_list = np.arange(0, 2, 1)

30


In [63]:
config_device = [xilinx_1]
config_sweep = [
    [
        [xilinx_1.sweep, 'reps', reps_list],
    ],
    [
        [xilinx_1.sweep, 'if_frequency_hz', if_frequency_list],
    ],
]

In [64]:
freqs = []
S = []

for f in if_frequency_list:
    # xilinx_1.sweep(if_frequency_hz=f)

    t_gen = np.array([0, 0.1, 0.8, 1.5, 1.6])*2
    s_gen = np.array([0, 1, 1, 1, 0])

    dr_readout1 = drx(soc=xilinx_1.soccfg,
                    dr_ch=dr_ch0, ro_ch=ro_ch0, frequency= f / 1e6, gain=1, phase=0)

    dr_readout1.wave.add(name='x2', t_data=t_gen, s_data=s_gen, idx=-1, interp_order=1)
    dr_readout1.rox.set(length=1.6, delay=0.4, sleep=2.8)
    xilinx_1.add(dr_readout1=dr_readout1)

    trace = xilinx_1.acquire_decimated()

    s = trace.mean(dim=["ticks"]).squeeze()

    freqs.append(f)
    S.append(s.values)

S = np.asarray(S).squeeze()

In [65]:
plt.figure()
plt.scatter(S.real, S.imag, c=freqs)
plt.axis("equal")

<IPython.core.display.Javascript object>

(-53.574576271186444,
 -35.19152542372881,
 -14.675084745762712,
 41.914632768361585)

In [12]:
_file_name = 'test_dds_nd.zarr'
file_name= data_dir / _file_name

do_sweep(config_device, config_sweep, xilinx_1.acquire_decimated, file_name)

# xilinx_1.soc.reset_gens()

  0%|          | 0/82 [00:00<?, ?it/s]

(0, 0) ((0,), (8080000000.0,))


  1%|          | 1/82 [00:01<02:06,  1.56s/it]

(0, 1) ((0,), (8080050000.0,))


  2%|▏         | 2/82 [00:02<01:35,  1.20s/it]

...

 98%|█████████▊| 80/82 [01:04<00:01,  1.17it/s]

(1, 39) ((1,), (8081950000.0,))


 99%|█████████▉| 81/82 [01:04<00:00,  1.47it/s]

(1, 40) ((1,), (8082000000.0,))


100%|██████████| 82/82 [01:05<00:00,  1.25it/s]


In [ ]:
_file_name = 'test_dds_nd.zarr'

with xr.open_zarr(data_dir / _file_name) as f:
    iq_mat = f['IQ decimated'].load()

fig = plt.figure(figsize=(12, 6))
gs = fig.add_gridspec(2, 2)

for _rox in iq_mat.rfsoc4x2_1_rox:
    ax0 = fig.add_subplot(gs[0,_rox])
    ax1 = fig.add_subplot(gs[1,_rox])

    s_data = iq_mat.sel(rfsoc4x2_1_rox=_rox).squeeze()
    # t_exp = s_data.tx

    s_int = s_data.mean(dim='rfsoc4x2_1_ticks')
    plot_if_frequency = s_int.rfsoc4x2_1_if_frequency_hz.data/1e9

    abs_data = xr.apply_ufunc(np.abs, s_int)
    abs_mean = abs_data.mean(dim='rfsoc4x2_1_reps')
    abs_std = abs_data.std(dim='rfsoc4x2_1_reps')

    ph_data = xr.apply_ufunc(np.angle,s_int)
    ph_data = xr.apply_ufunc(np.unwrap,ph_data)

    ph_mean = ph_data.mean(dim='rfsoc4x2_1_reps')
    ph_std = ph_data.std(dim='rfsoc4x2_1_reps')

    coef = np.polyfit(plot_if_frequency, np.unwrap(ph_mean), 1)
    ph_fit = np.polyval(coef, plot_if_frequency)

    ax0.plot(plot_if_frequency, 20*np.log10(abs_mean), '.-', color='C2')
    # ax0.fill_between(20*np.log10(abs_mean),
    #                  abs_mean-abs_std,
    #                  abs_mean+abs_std, alpha=0.3, color='C0')

    ax1.plot(plot_if_frequency, np.unwrap(ph_mean)-ph_fit, '.-', color='C2')
    ax1.fill_between(plot_if_frequency, np.unwrap(ph_mean)-ph_fit-ph_std, np.unwrap(ph_mean)-ph_fit+ph_std, alpha=0.3, color='C0')

# ax0.set_xlim(8.100, 8.1043)
# ax1.set_xlim(8.100, 8.1043)

plt.show()

In [ ]:
_file_name = 'test_dds_nd.zarr'

with xr.open_zarr(data_dir / _file_name) as f:
    iq_mat = f['IQ decimated'].load()

fig = plt.figure(figsize=(12, 6))
gs = fig.add_gridspec(2, 2)

for _rox in iq_mat.rfsoc4x2_1_rox:
    ax0 = fig.add_subplot(gs[0,_rox])
    ax1 = fig.add_subplot(gs[1,_rox])

    s_data = iq_mat.sel(rfsoc4x2_1_rox=_rox).squeeze()
    # t_exp = s_data.tx

    idx_if_frequency = 0
    s_int = s_data.isel(rfsoc4x2_1_if_frequency_hz=idx_if_frequency)

    for val_rfsoc4x2_1_reps in s_int.rfsoc4x2_1_reps.data:
        _s_int = s_int.sel(rfsoc4x2_1_reps=val_rfsoc4x2_1_reps).data
        ax0.scatter(np.real(_s_int), np.imag(_s_int))

    # idx_if_frequency = 100
    # s_int = s_data.isel(rfsoc4x2_1_if_frequency_hz=idx_if_frequency)
    #
    # for val_rfsoc4x2_1_reps in s_int.rfsoc4x2_1_reps.data:
    #     _s_int = s_int.sel(rfsoc4x2_1_reps=val_rfsoc4x2_1_reps).data
    #     ax1.scatter(np.real(_s_int), np.imag(_s_int))



    # for idx_if_frequency in s_data.rfsoc4x2_1_if_frequency_hz.data[:10]:
    #     # idx_if_frequency = 1
    #     s_int = s_data.sel(rfsoc4x2_1_if_frequency_hz=idx_if_frequency).mean()
    #     ax1.scatter(np.real(s_int), np.imag(s_int))

    plot_if_frequency = s_data.rfsoc4x2_1_if_frequency_hz.data

    s_int = s_data.mean(dim=['rfsoc4x2_1_reps', 'rfsoc4x2_1_ticks'])
    # coef = np.polyfit(plot_if_frequency, np.unwrap(np.angle(s_int)), 1)
    # ph_fit = np.polyval(coef, plot_if_frequency)
    #
    # ph_fit = xr.DataArray(
    #     ph_fit,
    #     dims=["rfsoc4x2_1_if_frequency_hz"],
    #     coords={
    #         "rfsoc4x2_1_if_frequency_hz": s_int.rfsoc4x2_1_if_frequency_hz
    #     },
    # )
    # s_int = s_int * np.exp(-1j * ph_fit)
    ax1.scatter(np.real(s_int), np.imag(s_int), c=plot_if_frequency)


    # for val_rfsoc4x2_1_ticks in s_int.rfsoc4x2_1_ticks.data:
    #     _s_int = s_int.sel(rfsoc4x2_1_ticks=val_rfsoc4x2_1_ticks).data
    #     ax1.scatter(np.real(_s_int), np.imag(_s_int))

#     plot_if_frequency = s_int.rfsoc4x2_1_if_frequency_hz.data/1e9
#
#     abs_data = xr.apply_ufunc(np.abs, s_int)
#     abs_mean = abs_data.mean(dim='rfsoc4x2_1_reps')
#     abs_std = abs_data.std(dim='rfsoc4x2_1_reps')
#
#     ph_data = xr.apply_ufunc(np.angle,s_int)
#     ph_data = xr.apply_ufunc(np.unwrap,ph_data)
#
#     ph_mean = ph_data.mean(dim='rfsoc4x2_1_reps')
#     ph_std = ph_data.std(dim='rfsoc4x2_1_reps')
#
#     coef = np.polyfit(plot_if_frequency, np.unwrap(ph_mean), 1)
#     ph_fit = np.polyval(coef, plot_if_frequency)
#
#     ax0.plot(plot_if_frequency, 20*np.log10(abs_mean), '.-', color='C2')
#     # ax0.fill_between(20*np.log10(abs_mean),
#     #                  abs_mean-abs_std,
#     #                  abs_mean+abs_std, alpha=0.3, color='C0')
#
#     ax1.plot(plot_if_frequency, np.unwrap(ph_mean)-ph_fit, '.-', color='C2')
#     ax1.fill_between(plot_if_frequency, np.unwrap(ph_mean)-ph_fit-ph_std, np.unwrap(ph_mean)-ph_fit+ph_std, alpha=0.3, color='C0')
#
# ax0.set_xlim(8.100, 8.1043)
# ax1.set_xlim(8.100, 8.1043)

ax0.set_aspect('equal')
ax1.set_aspect('equal')

ax0.scatter([0],[0], marker='+')
ax1.scatter([0],[0], marker='+')
plt.show()

In [ ]:
s = (
    s_data
    .isel(rfsoc4x2_1_reps=0)
    .isel(rfsoc4x2_1_ticks=slice(100, 200))
    .mean(dim="rfsoc4x2_1_ticks")
)

plt.figure()
plt.plot(plot_if_frequency, np.unwrap(np.angle(s)))

In [ ]:
# tmp = s_data.isel(rfsoc4x2_1_if_frequency_hz=2)

plt.figure()
tmp = s_data.isel(rfsoc4x2_1_if_frequency_hz=0)
plt.plot(np.real(tmp.sel(rfsoc4x2_1_reps=0)), alpha=0.2)
plt.plot(np.imag(tmp.sel(rfsoc4x2_1_reps=0)), alpha=0.2)


tmp = s_data.isel(rfsoc4x2_1_if_frequency_hz=1)
plt.plot(np.real(tmp.sel(rfsoc4x2_1_reps=0)), alpha=0.2)
plt.plot(np.imag(tmp.sel(rfsoc4x2_1_reps=0)), alpha=0.2)

tmp = s_data.isel(rfsoc4x2_1_if_frequency_hz=1)
plt.plot(np.real(tmp.sel(rfsoc4x2_1_reps=0)), alpha=0.2)
plt.plot(np.imag(tmp.sel(rfsoc4x2_1_reps=0)), alpha=0.2)

In [ ]:
18.63 * np.log10(30) - 63.53

In [ ]:
_file_name = 'test_dds_nd.zarr'

with xr.open_zarr(data_dir / _file_name) as f:
    iq_mat = f['IQ decimated'].load()

fig = plt.figure(figsize=(12, 6))
gs = fig.add_gridspec(2, 2)

for _rox in iq_mat.rfsoc4x2_1_rox:
    ax0 = fig.add_subplot(gs[0,_rox])
    ax1 = fig.add_subplot(gs[1,_rox])

    s_data = iq_mat.sel(rfsoc4x2_1_rox=_rox).squeeze()
    t_exp = s_data.tx

    idx_if_frequency = 0
    s_int = s_data.isel(rfsoc4x2_1_if_frequency_hz=idx_if_frequency)

    # for val_rfsoc4x2_1_reps in s_int.rfsoc4x2_1_reps.data:
    #     _s_int = s_int.sel(rfsoc4x2_1_reps=val_rfsoc4x2_1_reps).data
    #     ax0.scatter(np.real(_s_int), np.imag(_s_int))

    # idx_if_frequency = 100
    # s_int = s_data.isel(rfsoc4x2_1_if_frequency_hz=idx_if_frequency)
    #
    # for val_rfsoc4x2_1_reps in s_int.rfsoc4x2_1_reps.data:
    #     _s_int = s_int.sel(rfsoc4x2_1_reps=val_rfsoc4x2_1_reps).data
    #     ax1.scatter(np.real(_s_int), np.imag(_s_int))



    # for idx_if_frequency in s_data.rfsoc4x2_1_if_frequency_hz.data[:10]:
    #     # idx_if_frequency = 1
    #     s_int = s_data.sel(rfsoc4x2_1_if_frequency_hz=idx_if_frequency).mean()
    #     ax1.scatter(np.real(s_int), np.imag(s_int))

    plot_if_frequency = s_data.rfsoc4x2_1_if_frequency_hz.data

    s_int = s_data.mean(dim=['rfsoc4x2_1_reps', 'rfsoc4x2_1_ticks'])
    coef = np.polyfit(plot_if_frequency, np.unwrap(np.angle(s_int)), 1)
    ph_fit = np.polyval(coef, plot_if_frequency)

    ph_fit = xr.DataArray(
        ph_fit,
        dims=["rfsoc4x2_1_if_frequency_hz"],
        coords={
            "rfsoc4x2_1_if_frequency_hz": s_int.rfsoc4x2_1_if_frequency_hz
        },
    )
    s_int = s_int * np.exp(-1j * ph_fit)
    ax1.scatter(np.real(s_int), np.imag(s_int), c=plot_if_frequency)


    # for val_rfsoc4x2_1_ticks in s_int.rfsoc4x2_1_ticks.data:
    #     _s_int = s_int.sel(rfsoc4x2_1_ticks=val_rfsoc4x2_1_ticks).data
    #     ax1.scatter(np.real(_s_int), np.imag(_s_int))

#     plot_if_frequency = s_int.rfsoc4x2_1_if_frequency_hz.data/1e9
#
#     abs_data = xr.apply_ufunc(np.abs, s_int)
#     abs_mean = abs_data.mean(dim='rfsoc4x2_1_reps')
#     abs_std = abs_data.std(dim='rfsoc4x2_1_reps')
#
#     ph_data = xr.apply_ufunc(np.angle,s_int)
#     ph_data = xr.apply_ufunc(np.unwrap,ph_data)
#
#     ph_mean = ph_data.mean(dim='rfsoc4x2_1_reps')
#     ph_std = ph_data.std(dim='rfsoc4x2_1_reps')
#
#     coef = np.polyfit(plot_if_frequency, np.unwrap(ph_mean), 1)
#     ph_fit = np.polyval(coef, plot_if_frequency)
#
#     ax0.plot(plot_if_frequency, 20*np.log10(abs_mean), '.-', color='C2')
#     # ax0.fill_between(20*np.log10(abs_mean),
#     #                  abs_mean-abs_std,
#     #                  abs_mean+abs_std, alpha=0.3, color='C0')
#
#     ax1.plot(plot_if_frequency, np.unwrap(ph_mean)-ph_fit, '.-', color='C2')
#     ax1.fill_between(plot_if_frequency, np.unwrap(ph_mean)-ph_fit-ph_std, np.unwrap(ph_mean)-ph_fit+ph_std, alpha=0.3, color='C0')
#
# ax0.set_xlim(8.100, 8.1043)
# ax1.set_xlim(8.100, 8.1043)

ax0.set_aspect('equal')
ax1.set_aspect('equal')

ax0.scatter([0],[0], marker='+')
ax1.scatter([0],[0], marker='+')
plt.show()

In [ ]:
_file_name = 'test_dds_nd.zarr'

with xr.open_zarr(data_dir / _file_name) as f:
    iq_mat = f['IQ decimated'].load()

fig = plt.figure(figsize=(12, 6))
gs = fig.add_gridspec(2, 2)

for _rox in iq_mat.rfsoc4x2_1_rox:
    ax0 = fig.add_subplot(gs[0,_rox])
    ax1 = fig.add_subplot(gs[1,_rox])

    s_data = iq_mat.sel(rfsoc4x2_1_rox=_rox).squeeze()
    t_exp = s_data.tx

    s_int = s_data.mean(dim='rfsoc4x2_1_reps')
    plot_if_frequency = s_int.rfsoc4x2_1_if_frequency_hz.data/1e9

    # ph_int = np.unwrap(np.angle(s_int.mean(dim='rfsoc4x2_1_ticks')))
    # coef = np.polyfit(plot_if_frequency, np.unwrap(ph_int), 1)
    # ph_fit = np.polyval(coef, plot_if_frequency)
    #
    # ph_fit = xr.DataArray(
    #     ph_fit,
    #     dims=["rfsoc4x2_1_if_frequency_hz"],
    #     coords={
    #         "rfsoc4x2_1_if_frequency_hz": s_int.rfsoc4x2_1_if_frequency_hz
    #     },
    # )
    #
    # s_int = s_int * np.exp(-1j * ph_fit)

    # ax0.plot(plot_if_frequency, np.real(s_int.mean(dim='rfsoc4x2_1_ticks')))
    # ax1.plot(plot_if_frequency, np.imag(s_int.mean(dim='rfsoc4x2_1_ticks')))

    abs_data = xr.apply_ufunc(np.abs, s_int)
    abs_mean = abs_data.mean(dim='rfsoc4x2_1_ticks')
    abs_std = abs_data.std(dim='rfsoc4x2_1_ticks')

    ph_data = xr.apply_ufunc(np.angle,s_int)
    ph_data = xr.apply_ufunc(np.unwrap,ph_data)
    ph_mean = ph_data.mean(dim='rfsoc4x2_1_ticks')
    ph_std = ph_data.mean(dim='rfsoc4x2_1_ticks')

    coef = np.polyfit(plot_if_frequency, np.unwrap(ph_mean), 1)
    ph_fit = np.polyval(coef, plot_if_frequency)

    ax0.plot(plot_if_frequency, 20*np.log10(abs_mean), '.-', color='C2')
    # ax0.fill_between(plot_if_frequency,
    #                  abs_mean-abs_std,
    #                  abs_mean+abs_std, alpha=0.3, color='C0')

    ax1.plot(plot_if_frequency, np.unwrap(ph_mean)-ph_fit, '.-', color='C2')
    ax1.fill_between(plot_if_frequency, np.unwrap(ph_mean)-ph_fit-ph_std, np.unwrap(ph_mean)-ph_fit+ph_std, alpha=0.3, color='C0')

plt.show()